# Festa em Cena — análise exploratória de pedidos de Halloween

Projeto do Módulo 1 (SCTEC). Objetivo: ler, limpar e explorar `halloween_vendas.csv`
antes de comparar canais e períodos de venda.

In [1]:
#Bibliotecas usadas no projeto

import re                                               # Expressões regulares (limpeza de texto)
from datetime import datetime                           # Conversão de datas
from pathlib import Path                                # Caminho de arquivos
import pandas as pd                                     # Tabelas (DataFrames)

In [2]:
def ler_csv(caminho):
    """Lê o CSV e devolve um DataFrame com todas as colunas como texto.

    Lemos tudo como texto (dtype=str) para não deixar o Pandas
    adivinhar tipos: assim conseguimos ver valores problemáticos
    como eles vieram e convertê-los depois, com regras.
    """
    arquivo = Path(caminho)
    print("Pasta de execução:", Path.cwd())
    print("Arquivo encontrado:", arquivo.exists())
    df = pd.read_csv(arquivo, sep=",", encoding="utf-8", dtype=str)
    return df

In [3]:
CAMINHO_CSV = "halloween_vendas.csv" #Chama a função e guarda o que ela devolveu na variável

df_original = ler_csv(CAMINHO_CSV) #Usamos o nome “original” porque nunca vamos alterar essa tabela: o PDF exige preservar o original.

print("Total de linhas e colunas:", df_original.shape) # mostra (linhas, colunas). Antes de executar, tente prever: o CSV tem 1.501 linhas de texto, contando o cabeçalho. Quantas linhas o Pandas vai mostrar?
df_original.head(3) # Mostra os 3 primeiros registros, como o Sprint 1 pede.

Pasta de execução: c:\Users\N-lenovo\Documents\SCTEC\festa-em-cena
Arquivo encontrado: True
Total de linhas e colunas: (1500, 14)


,pedido_id,data_venda,produto,categoria,canal_venda,cidade,quantidade,preco_unitario,custo_unitario,desconto_pct,status_pedido,publico,tamanho,data_entrega
0,844,2025-11-21 18:02:35,Fantasia de vampiro,Fantasia,Loja física,Curitiba,3,132.5,72.08,15,Cancelado,Adulto,G,NaN
1,378,2024-10-29 18:41:26,Fantasia de bruxa,FANTASIA!!!,Loja física,Maringá,3,110,57,5,Concluído,Adulto,M,2024-11-03 18:41:26
2,953,2025-09-05 21:16:09,Fantasia de bruxa,Fantasia,Web,Maringá,2,116.6,60.42,5,Concluído,Adulto,M,2025-09-10 21:16:09


Para cada coluna, isso mostra o nome, quantos valores estão preenchidos (non-null) e o tipo. Olhar com atenção para as colunas que têm menos preenchidos que o total de linhas: esses serão os nossos primeiros suspeitos de dados ausentes.

In [4]:
df_original.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   pedido_id       1500 non-null   str  
 1   data_venda      1500 non-null   str  
 2   produto         1500 non-null   str  
 3   categoria       1482 non-null   str  
 4   canal_venda     1500 non-null   str  
 5   cidade          1500 non-null   str  
 6   quantidade      1500 non-null   str  
 7   preco_unitario  1488 non-null   str  
 8   custo_unitario  1493 non-null   str  
 9   desconto_pct    1500 non-null   str  
 10  status_pedido   1500 non-null   str  
 11  publico         1500 non-null   str  
 12  tamanho         1500 non-null   str  
 13  data_entrega    1357 non-null   str  
dtypes: str(14)
memory usage: 164.2 KB


In [5]:
# Conta quantos valores vazios existem em cada coluna
ausentes = df_original.isna().sum()
print(ausentes)

pedido_id           0
data_venda          0
produto             0
categoria          18
canal_venda         0
cidade              0
quantidade          0
preco_unitario     12
custo_unitario      7
desconto_pct        0
status_pedido       0
publico             0
tamanho             0
data_entrega      143
dtype: int64


## Primeira inspeção

- A base tem 1.500 linhas e 14 colunas.
- Colunas com valores vazios: `categoria` (18), `preco_unitario` (12),
  `custo_unitario` (7) e `data_entrega` (143).
- O `info()` só detecta células vazias; valores preenchidos mas inválidos
  (ex.: quantidade 0 ou data escrita errada) serão investigados nos próximos passos.

- Cada linha representa um pedido de um único produto. A coluna `quantidade`
  informa quantas unidades desse produto foram pedidas: uma linha com
  quantidade 3 conta como 1 pedido e 3 unidades.

In [6]:
# Como os textos vieram no arquivo, antes de qualquer limpeza
print(df_original["categoria"].value_counts(dropna=False))
print()
print(df_original["canal_venda"].value_counts(dropna=False))

categoria
Fantasia            892
Acessorio           459
 Fantasia            48
  FANTASIA!!!        39
 Acessorio           28
NaN                  18
  ACESSORIO!!!       16
Name: count, dtype: int64

canal_venda
Web              646
Loja física      535
Mercado          274
 web              20
 loja física      16
 mercado           9
Name: count, dtype: int64


## Sprint 2 — padronização de textos

In [7]:
def limpar_categoria(valor):
    """Limpa um valor da coluna categoria e devolve o texto padronizado.

    Regras:
    1. Vazio (NaN) vira "sem categoria" (exigência do projeto).
    2. strip() tira espaços das pontas; lower() deixa tudo minúsculo.
    3. re.sub() apaga pontuação, mantendo apenas letras e espaços
       (as categorias deste CSV não usam acentos).
    4. Espaços repetidos no meio viram um só.
    5. Se sobrar texto vazio (ex.: "!!!"), também vira "sem categoria".
    6. A primeira letra volta a ser maiúscula, só para apresentação
       (ex.: "fantasia" -> "Fantasia").
    """
    if pd.isna(valor):                              # regra 1
        return "sem categoria"

    texto = valor.strip().lower()                   # regra 2
    texto = re.sub(r"[^a-z\s]", "", texto)          # regra 3
    texto = re.sub(r"\s+", " ", texto).strip()      # regra 4

    if texto == "":                                 # regra 5
        return "sem categoria"
    else:
        return texto.capitalize()                   # regra 6

In [8]:
# Testa a função com exemplos tirados do value_counts
exemplos = ["Fantasia", " Fantasia", "  FANTASIA!!!", "  ACESSORIO!!!", None, "!!!"]

for exemplo in exemplos:
    print(repr(exemplo), "->", repr(limpar_categoria(exemplo)))

'Fantasia' -> 'Fantasia'
' Fantasia' -> 'Fantasia'
'  FANTASIA!!!' -> 'Fantasia'
'  ACESSORIO!!!' -> 'Acessorio'
None -> 'sem categoria'
'!!!' -> 'sem categoria'


In [9]:
# Cópia de trabalho: o df_original nunca é alterado
df_org = df_original.copy()

# Guarda o texto original para mostrar "antes e depois"
df_org["categoria_original"] = df_org["categoria"]

# Aplica a função em cada linha da coluna
df_org["categoria"] = df_org["categoria"].apply(limpar_categoria)

print(df_org["categoria"].value_counts(dropna=False))

categoria
Fantasia         979
Acessorio        503
sem categoria     18
Name: count, dtype: int64


In [10]:
# 1) Vazias que foram preenchidas com "sem categoria"
vazias_preenchidas = df_org["categoria"] == "sem categoria"

# 2) Preenchidas no original, mas que mudaram com a limpeza
nao_vazias_alteradas = (
    df_org["categoria_original"].notna()
    & (df_org["categoria_original"] != df_org["categoria"])
)

print("Categorias vazias preenchidas:", vazias_preenchidas.sum())
print("Categorias não vazias alteradas:", nao_vazias_alteradas.sum())

# Exemplos de antes e depois (um de cada tipo)
print(df_org.loc[nao_vazias_alteradas, ["categoria_original", "categoria"]].drop_duplicates())

Categorias vazias preenchidas: 18
Categorias não vazias alteradas: 131
   categoria_original  categoria
1       FANTASIA!!!     Fantasia
12          Fantasia    Fantasia
18     ACESSORIO!!!    Acessorio
22         Acessorio   Acessorio


In [11]:
# Tradução: texto limpo (minúsculo, sem espaços) -> nome oficial do canal
MAPA_CANAIS = {
    "web": "Web",
    "loja física": "Loja física",
    "mercado": "Mercado",
}

def padronizar_canal(valor):
    """Devolve o nome oficial do canal, ou None se o canal for desconhecido.

    Um canal desconhecido NÃO é adivinhado: vira vazio para ser investigado.
    """
    if pd.isna(valor):
        return None
    chave = valor.strip().lower()
    if chave in MAPA_CANAIS:
        return MAPA_CANAIS[chave]
    else:
        return None

df_org["canal_original"] = df_org["canal_venda"]
df_org["canal_venda"] = df_org["canal_venda"].apply(padronizar_canal)

canais_alterados = df_org["canal_original"] != df_org["canal_venda"]
print(df_org["canal_venda"].value_counts(dropna=False))
print("Canais alterados:", canais_alterados.sum())
print("Canais desconhecidos:", df_org["canal_venda"].isna().sum())

canal_venda
Web            666
Loja física    551
Mercado        283
Name: count, dtype: int64
Canais alterados: 45
Canais desconhecidos: 0


### Resultado da padronização

- **Categoria:** apliquei `strip()`, `lower()` e `re.sub(r"[^a-z\s]", "", texto)`
  para retirar espaços, maiúsculas e pontuação. Vazios viraram "sem categoria".
  - Vazias preenchidas: 18
  - Não vazias alteradas: 131 (ex.: "  FANTASIA!!!" → "Fantasia")
  - Resultado: Fantasia 979, Acessorio 503, sem categoria 18 (total 1.500)
- **Canal:** `strip()` + `lower()` + dicionário de tradução para os nomes oficiais.
  - Canais alterados: 45 (ex.: " web" → "Web"). Canais desconhecidos: 0.
- Os textos originais ficaram nas colunas `categoria_original` e `canal_original`.

## Sprint 3 — validação de quantidade e preço

In [12]:
# Todos os valores diferentes de quantidade, como vieram no arquivo
print(df_org["quantidade"].value_counts(dropna=False))

quantidade
2     462
3     429
1     416
6      54
7      46
4      44
5      42
-1      7
Name: count, dtype: int64


In [13]:
def eh_numero(texto):
    """Devolve True se o texto puder ser convertido em número."""
    try:
        float(texto)          # tenta converter
        return True           # se chegou aqui, deu certo
    except ValueError:
        return False          # se deu erro, não é número

# Teste rápido
for exemplo in ["12.50", "-3", "abc", "12,50"]:
    print(repr(exemplo), "->", eh_numero(exemplo))

'12.50' -> True
'-3' -> True
'abc' -> False
'12,50' -> False


In [14]:
precos = df_org["preco_unitario"]

# Preenchidos, mas que não viram número
nao_numericos = precos.notna() & ~precos.apply(eh_numero)
print("Preços vazios:", precos.isna().sum())
print("Preços preenchidos que não são número:", nao_numericos.sum())
print(precos[nao_numericos].value_counts())

# Entre os que são número: quantos são negativos?
numericos = precos[precos.notna() & ~nao_numericos].astype(float)
print("Preços negativos:", (numericos < 0).sum())
print("Menor preço:", numericos.min(), "| Maior preço:", numericos.max())

Preços vazios: 12
Preços preenchidos que não são número: 0
Series([], Name: count, dtype: int64)
Preços negativos: 0
Menor preço: 22.0 | Maior preço: 132.5


In [15]:
def converter_quantidade(texto):
    """Converte quantidade para inteiro. Devolve None se for inválida.

    Regra do projeto: quantidade deve ser um número inteiro maior que zero.
    Valores inválidos (vazio, texto, decimal, zero ou negativo) não são
    corrigidos nem inventados: ficam vazios e o pedido é marcado com problema.
    """
    if pd.isna(texto):
        return None
    try:
        numero = float(texto)
    except ValueError:
        return None                      # não é número: continua sem parar o programa
    if numero.is_integer() and numero > 0:
        return int(numero)
    else:
        return None


def converter_preco(texto):
    """Converte preço para número decimal. Devolve None se for inválido.

    Regra do projeto: preço deve estar preenchido e não pode ser negativo.
    Preço vazio NÃO vira zero: vazio significa "desconhecido", e zero seria
    um valor inventado.
    """
    if pd.isna(texto):
        return None
    try:
        numero = float(texto)
    except ValueError:
        return None
    if numero >= 0:
        return numero
    else:
        return None

In [16]:
# Colunas convertidas (o texto original continua em "quantidade" e "preco_unitario")
df_org["quantidade_num"] = df_org["quantidade"].apply(converter_quantidade).astype("Int64")
df_org["preco_num"] = df_org["preco_unitario"].apply(converter_preco)

# Marcas (flags) de problema: True = o pedido tem problema nesse campo
df_org["problema_quantidade"] = df_org["quantidade_num"].isna()
df_org["problema_preco"] = df_org["preco_num"].isna()

# O pedido tem PELO MENOS UM problema numérico?
df_org["problema_numerico"] = df_org["problema_quantidade"] | df_org["problema_preco"]

In [17]:
# Contagem por campo, usando um dicionário e um laço
problemas_por_campo = {
    "quantidade": df_org["problema_quantidade"].sum(),
    "preco_unitario": df_org["problema_preco"].sum(),
}
for campo, total in problemas_por_campo.items():
    print(f"Problemas em {campo}: {total}")

# Contagem por pedido (sem somar sobreposições)
ambos = df_org["problema_quantidade"] & df_org["problema_preco"]
print(f"Pedidos com problema nos dois campos: {ambos.sum()}")
print(f"Total de pedidos afetados: {df_org['problema_numerico'].sum()}")

# Exemplos dos registros com problema
print(df_org.loc[df_org["problema_numerico"], ["pedido_id", "quantidade", "preco_unitario"]].head())

Problemas em quantidade: 7
Problemas em preco_unitario: 12
Pedidos com problema nos dois campos: 0
Total de pedidos afetados: 19
    pedido_id quantidade preco_unitario
23       1500          3            NaN
164       630         -1             22
199       625          2            NaN
216      1000          1            NaN
239      1250          1            NaN


In [18]:
# Contagem por campo, usando um dicionário e um laço
problemas_por_campo = {
    "quantidade": df_org["problema_quantidade"].sum(),
    "preco_unitario": df_org["problema_preco"].sum(),
}
for campo, total in problemas_por_campo.items():
    print(f"Problemas em {campo}: {total}")

# Contagem por pedido (sem somar sobreposições)
ambos = df_org["problema_quantidade"] & df_org["problema_preco"]
print(f"Pedidos com problema nos dois campos: {ambos.sum()}")
print(f"Total de pedidos afetados: {df_org['problema_numerico'].sum()}")

# Exemplos dos registros com problema
print(df_org.loc[df_org["problema_numerico"], ["pedido_id", "quantidade", "preco_unitario"]].head())

Problemas em quantidade: 7
Problemas em preco_unitario: 12
Pedidos com problema nos dois campos: 0
Total de pedidos afetados: 19
    pedido_id quantidade preco_unitario
23       1500          3            NaN
164       630         -1             22
199       625          2            NaN
216      1000          1            NaN
239      1250          1            NaN


### Resultado da validação

- **Regra da quantidade:** inteiro maior que zero. 7 pedidos têm quantidade `-1`.
- **Regra do preço:** preenchido e não negativo. 12 pedidos têm preço vazio;
  nenhum é negativo ou não numérico.
- Nenhum pedido tem os dois problemas, por isso o total de pedidos afetados é 19.
- Os registros **não foram apagados nem corrigidos**: ficaram marcados nas colunas
  `problema_quantidade`, `problema_preco` e `problema_numerico`.
- Preço vazio não virou zero, porque zero seria um valor inventado.
- Usei `try/except` para que um valor impossível de converter não interrompa o programa.